In [6]:
import sys
import json
import torch
from pathlib import Path

BASE = Path("/workspace/tiny-llm-from-scratch")
sys.path.append(str(BASE))

from tokenizer.char_tokenizer import CharTokenizer
from model.tiny_transformer import GPTConfig, TinyGPT, count_parameters

torch.set_num_threads(4)

tokenizer = CharTokenizer.load(BASE / "tokenizer/tokenizer_char.json")

train_text = (BASE / "data/train.txt").read_text(encoding="utf-8")
val_text = (BASE / "data/val.txt").read_text(encoding="utf-8")

train_ids = torch.tensor(tokenizer.encode(train_text), dtype=torch.long)
val_ids = torch.tensor(tokenizer.encode(val_text), dtype=torch.long)

config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=64,
    n_layer=1,
    n_head=2,
    n_embd=32,
    dropout=0.1
)

model = TinyGPT(config)

print("Vocab size:", tokenizer.vocab_size)
print("Parameters:", count_parameters(model))
print("Train tokens:", len(train_ids))

Vocab size: 44
Parameters: 16224
Train tokens: 401850


In [7]:
def get_batch(data, batch_size, block_size):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(100):
    xb, yb = get_batch(train_ids, batch_size=8, block_size=64)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print("step:", step, "loss:", loss.item())

step: 0 loss: 3.792194366455078
step: 25 loss: 3.509941339492798
step: 50 loss: 3.382021903991699
step: 75 loss: 3.2380454540252686
